# Fourier-domain mu and beta extraction for every EEG recording

This notebook discovers every `.gdf` file below `data/processed/Signals` (closed-eye/open-eye baselines, acquisition runs, and online trials), processes one recording at a time, and saves its frequency-domain EEG data.

Rather than taking one FFT over an entire non-stationary recording, it uses **Welch's method**, which averages overlapping windowed FFTs and gives a substantially more stable power spectral density (PSD). The retained bands are:

- **Mu:** 8 Hz to <13 Hz
- **Beta:** 13 Hz to 30 Hz

EOG and EMG channels are excluded. For each source file, the notebook saves (1) the channel-by-frequency PSD between 8 and 30 Hz and (2) compact per-channel mu/beta features. Processing is checkpointed and resumable because the complete GDF collection is large.

In [ ]:
from pathlib import Path
import re
import time
import warnings

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display

mne.set_log_level('ERROR')

# Frequency definitions and Welch settings.
MU_RANGE_HZ = (8.0, 13.0)       # upper endpoint excluded
BETA_RANGE_HZ = (13.0, 30.0)    # upper endpoint included
PSD_RANGE_HZ = (1.0, 40.0)      # denominator for relative power
WELCH_WINDOW_SECONDS = 4.0
WELCH_OVERLAP = 0.5
NON_SCALP_CHANNELS = {'EOG1', 'EOG2', 'EOG3', 'EMGg', 'EMGd'}

def find_signals_root(start=Path.cwd()):
    """Find data/processed/Signals from the notebook directory or a parent."""
    for parent in (start.resolve(), *start.resolve().parents):
        candidate = parent / 'data' / 'processed' / 'Signals'
        if candidate.is_dir():
            return candidate
    raise FileNotFoundError('Could not locate data/processed/Signals from the current directory.')

SIGNALS_ROOT = find_signals_root()
OUTPUT_ROOT = SIGNALS_ROOT.parent / 'mu_beta_fourier'
SPECTRA_ROOT = OUTPUT_ROOT / 'spectra'
FEATURES_ROOT = OUTPUT_ROOT / 'features_by_file'
MASTER_FEATURE_PATH = OUTPUT_ROOT / 'mu_beta_features_all_gdf.csv'
PROCESSING_LOG_PATH = OUTPUT_ROOT / 'processing_log.csv'

for directory in (SPECTRA_ROOT, FEATURES_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

gdf_files = sorted(SIGNALS_ROOT.rglob('*.gdf'), key=lambda p: p.as_posix().lower())
if not gdf_files:
    raise FileNotFoundError(f'No GDF files found below {SIGNALS_ROOT}')

print(f'Signals root: {SIGNALS_ROOT}')
print(f'Output root:  {OUTPUT_ROOT}')
print(f'GDF files discovered: {len(gdf_files):,}')

In [ ]:
def recording_metadata(path):
    """Parse dataset, participant, and recording type from the repository layout/name."""
    relative = path.relative_to(SIGNALS_ROOT)
    name = path.stem
    lower = name.lower()

    if lower.endswith('_ce_baseline'):
        recording, family = 'CE_baseline', 'baseline'
    elif lower.endswith('_oe_baseline'):
        recording, family = 'OE_baseline', 'baseline'
    elif re.search(r'_r1_acquisition$', lower):
        recording, family = 'R1_acquisition', 'acquisition'
    elif re.search(r'_r2_acquisition$', lower):
        recording, family = 'R2_acquisition', 'acquisition'
    elif match := re.search(r'_(r[3-6])_onlinet$', lower):
        recording, family = f'{match.group(1).upper()}_onlineT', 'online'
    else:
        recording, family = 'unclassified', 'unclassified'

    return {
        'source_file': relative.as_posix(),
        'dataset': path.parent.parent.name if len(relative.parts) >= 3 else '',
        'subject': path.parent.name,
        'recording': recording,
        'recording_family': family,
    }

inventory = pd.DataFrame([recording_metadata(path) for path in gdf_files])
inventory_summary = (
    inventory.groupby(['recording_family', 'recording'], dropna=False)
    .size().rename('files').reset_index()
)
display(inventory_summary)

if (inventory['recording_family'] == 'unclassified').any():
    display(inventory.loc[inventory['recording_family'] == 'unclassified'])
    raise ValueError('Some GDF filenames could not be classified; update recording_metadata().')

In [ ]:
def output_paths(gdf_path):
    """Preserve source subdirectories so identically named files cannot collide."""
    relative_stem = gdf_path.relative_to(SIGNALS_ROOT).with_suffix('')
    spectrum_path = SPECTRA_ROOT / relative_stem.parent / f'{relative_stem.name}_mu_beta.npz'
    feature_path = FEATURES_ROOT / relative_stem.parent / f'{relative_stem.name}_mu_beta_features.csv'
    return spectrum_path, feature_path

def outputs_are_valid(gdf_path):
    spectrum_path, feature_path = output_paths(gdf_path)
    if not spectrum_path.is_file() or not feature_path.is_file():
        return False
    try:
        with np.load(spectrum_path, allow_pickle=False) as saved:
            required = {'channels', 'frequencies_hz', 'psd_uv2_per_hz', 'mu_mask', 'beta_mask'}
            if not required.issubset(saved.files):
                return False
            if saved['psd_uv2_per_hz'].shape != (len(saved['channels']), len(saved['frequencies_hz'])):
                return False
        features = pd.read_csv(feature_path)
        return len(features) > 0 and set(features['band']) == {'mu', 'beta'}
    except Exception:
        return False

def process_gdf(gdf_path, overwrite=False):
    """Compute Welch (windowed FFT) PSD and save isolated mu/beta data and features."""
    spectrum_path, feature_path = output_paths(gdf_path)
    if not overwrite and outputs_are_valid(gdf_path):
        return {'source_file': recording_metadata(gdf_path)['source_file'], 'status': 'skipped', 'error': ''}

    spectrum_path.parent.mkdir(parents=True, exist_ok=True)
    feature_path.parent.mkdir(parents=True, exist_ok=True)
    metadata = recording_metadata(gdf_path)

    # preload=False prevents an entire recording (and certainly the dataset) being held in memory.
    raw = mne.io.read_raw_gdf(gdf_path, preload=False, verbose='ERROR')
    scalp_channels = [name for name in raw.ch_names if name not in NON_SCALP_CHANNELS]
    if not scalp_channels:
        raise ValueError('No scalp EEG channels remain after excluding EOG/EMG channels.')

    sfreq = float(raw.info['sfreq'])
    n_per_seg = min(int(round(WELCH_WINDOW_SECONDS * sfreq)), raw.n_times)
    n_fft = n_per_seg
    n_overlap = int(round(n_per_seg * WELCH_OVERLAP))

    spectrum = raw.compute_psd(
        method='welch', fmin=PSD_RANGE_HZ[0], fmax=PSD_RANGE_HZ[1],
        picks=scalp_channels, n_fft=n_fft, n_per_seg=n_per_seg,
        n_overlap=n_overlap, average='mean', reject_by_annotation=True,
        verbose=False,
    )
    psd_v2_per_hz, frequencies = spectrum.get_data(return_freqs=True)
    psd_uv2_per_hz = psd_v2_per_hz * 1e12

    mu_full = (frequencies >= MU_RANGE_HZ[0]) & (frequencies < MU_RANGE_HZ[1])
    beta_full = (frequencies >= BETA_RANGE_HZ[0]) & (frequencies <= BETA_RANGE_HZ[1])
    isolated = mu_full | beta_full
    if not mu_full.any() or not beta_full.any():
        raise ValueError(f'Frequency resolution did not produce both bands (sfreq={sfreq:g} Hz).')

    isolated_frequencies = frequencies[isolated]
    isolated_psd = psd_uv2_per_hz[:, isolated]
    mu_mask = (isolated_frequencies >= MU_RANGE_HZ[0]) & (isolated_frequencies < MU_RANGE_HZ[1])
    beta_mask = (isolated_frequencies >= BETA_RANGE_HZ[0]) & (isolated_frequencies <= BETA_RANGE_HZ[1])

    # Atomic replacements keep interrupted runs from leaving apparently complete outputs.
    spectrum_tmp = spectrum_path.with_name(spectrum_path.name + '.tmp.npz')
    np.savez_compressed(
        spectrum_tmp,
        channels=np.asarray(scalp_channels),
        frequencies_hz=isolated_frequencies,
        psd_uv2_per_hz=isolated_psd,
        mu_mask=mu_mask,
        beta_mask=beta_mask,
        source_file=np.asarray(metadata['source_file']),
        sfreq_hz=np.asarray(sfreq),
        duration_seconds=np.asarray(raw.n_times / sfreq),
        method=np.asarray('Welch PSD (overlapping windowed FFTs)'),
    )
    spectrum_tmp.replace(spectrum_path)

    total_power = np.trapezoid(psd_uv2_per_hz, frequencies, axis=1)
    feature_frames = []
    for band, band_mask, low_hz, high_hz in (
        ('mu', mu_full, *MU_RANGE_HZ),
        ('beta', beta_full, *BETA_RANGE_HZ),
    ):
        band_frequencies = frequencies[band_mask]
        band_psd = psd_uv2_per_hz[:, band_mask]
        band_power = np.trapezoid(band_psd, band_frequencies, axis=1)
        peak_frequency = band_frequencies[np.argmax(band_psd, axis=1)]
        frame = pd.DataFrame({
            **metadata,
            'channel': scalp_channels,
            'band': band,
            'band_low_hz': low_hz,
            'band_high_hz': high_hz,
            'band_power_uv2': band_power,
            'relative_power_1_40': np.divide(
                band_power, total_power,
                out=np.full_like(band_power, np.nan), where=total_power > 0,
            ),
            'peak_frequency_hz': peak_frequency,
            'mean_psd_uv2_per_hz': band_psd.mean(axis=1),
            'sfreq_hz': sfreq,
            'duration_seconds': raw.n_times / sfreq,
            'n_scalp_channels': len(scalp_channels),
        })
        feature_frames.append(frame)

    features = pd.concat(feature_frames, ignore_index=True)
    feature_tmp = feature_path.with_name(feature_path.name + '.tmp.csv')
    features.to_csv(feature_tmp, index=False)
    feature_tmp.replace(feature_path)
    raw.close()
    return {'source_file': metadata['source_file'], 'status': 'processed', 'error': ''}

print('Processing functions are ready.')

## Representative smoke test

This checks the transformation on one baseline, one acquisition, and one online recording before starting the complete batch. These outputs are valid checkpoints and will be skipped by the next cell.

In [ ]:
smoke_files = []
for family in ('baseline', 'acquisition', 'online'):
    candidates = inventory.index[inventory['recording_family'].eq(family)].tolist()
    if candidates:
        smoke_files.append(gdf_files[candidates[0]])

smoke_results = []
for path in smoke_files:
    started = time.perf_counter()
    try:
        result = process_gdf(path, overwrite=False)
        result['seconds'] = time.perf_counter() - started
    except Exception as exc:
        result = {
            'source_file': path.relative_to(SIGNALS_ROOT).as_posix(),
            'status': 'failed', 'error': f'{type(exc).__name__}: {exc}',
            'seconds': time.perf_counter() - started,
        }
    smoke_results.append(result)

smoke_log = pd.DataFrame(smoke_results)
display(smoke_log)
if smoke_log['status'].eq('failed').any():
    raise RuntimeError('A representative smoke test failed; inspect the table before full processing.')

## Process every GDF file

`RUN_FULL_BATCH` is deliberately `True`. Existing valid outputs are skipped, so rerunning the cell resumes rather than restarts. Set `OVERWRITE=True` only when changing the spectral parameters and intentionally rebuilding every output.

In [ ]:
RUN_FULL_BATCH = True
OVERWRITE = False

batch_results = []
if RUN_FULL_BATCH:
    batch_started = time.perf_counter()
    for number, path in enumerate(gdf_files, start=1):
        started = time.perf_counter()
        try:
            result = process_gdf(path, overwrite=OVERWRITE)
        except Exception as exc:
            result = {
                'source_file': path.relative_to(SIGNALS_ROOT).as_posix(),
                'status': 'failed',
                'error': f'{type(exc).__name__}: {exc}',
            }
        result['seconds'] = time.perf_counter() - started
        batch_results.append(result)

        if result['status'] == 'failed' or number == 1 or number % 10 == 0 or number == len(gdf_files):
            elapsed_min = (time.perf_counter() - batch_started) / 60
            print(f'[{number:>4}/{len(gdf_files)}] {result["status"]:<9} {result["source_file"]} ({elapsed_min:.1f} min elapsed)')

    batch_log = pd.DataFrame(batch_results)
    batch_log.to_csv(PROCESSING_LOG_PATH, index=False)
    display(batch_log['status'].value_counts().rename_axis('status').to_frame('files'))

    failures = batch_log.loc[batch_log['status'].eq('failed')]
    if not failures.empty:
        display(failures)
        raise RuntimeError(f'{len(failures)} GDF file(s) failed. Fix them and rerun; completed files will be skipped.')
else:
    print('Full batch disabled; only the representative smoke-test outputs exist.')

## Consolidate and verify complete coverage

This cell refuses to create a misleading master table if even one discovered GDF lacks valid output.

In [ ]:
coverage = inventory.copy()
coverage['output_valid'] = [outputs_are_valid(path) for path in gdf_files]
display(coverage.groupby(['recording_family', 'recording'])['output_valid'].agg(['sum', 'count']))

missing_outputs = coverage.loc[~coverage['output_valid']]
if not missing_outputs.empty:
    display(missing_outputs)
    raise RuntimeError(f'{len(missing_outputs)} of {len(gdf_files)} GDF files do not yet have valid mu/beta outputs.')

feature_paths = [output_paths(path)[1] for path in gdf_files]
all_features = pd.concat((pd.read_csv(path) for path in feature_paths), ignore_index=True)
all_features.to_csv(MASTER_FEATURE_PATH, index=False)

expected_rows = sum(
    len(pd.read_csv(path, usecols=['channel'])) for path in feature_paths
)
assert len(all_features) == expected_rows
assert all_features['source_file'].nunique() == len(gdf_files)
assert set(all_features['band']) == {'mu', 'beta'}

print(f'Complete: {len(gdf_files):,}/{len(gdf_files):,} GDF files represented.')
print(f'Feature rows: {len(all_features):,}')
print(f'Master features: {MASTER_FEATURE_PATH}')
print(f'Per-file spectra: {SPECTRA_ROOT}')

## Quality-control summaries

Power is strongly right-skewed, so plots use a log scale. Relative power is often more comparable between participants than absolute power, but both should be inspected for artifacts.

In [ ]:
summary = (
    all_features.groupby(['recording_family', 'recording', 'band'])
    .agg(
        files=('source_file', 'nunique'),
        median_band_power_uv2=('band_power_uv2', 'median'),
        median_relative_power=('relative_power_1_40', 'median'),
        median_peak_hz=('peak_frequency_hz', 'median'),
    )
    .reset_index()
)
display(summary)

plot_data = all_features.copy()
plot_data['log10_band_power_uv2'] = np.log10(plot_data['band_power_uv2'].clip(lower=np.finfo(float).tiny))
families = ['baseline', 'acquisition', 'online']
bands = ['mu', 'beta']

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for axis, band in zip(axes, bands):
    values = [
        plot_data.loc[
            plot_data['band'].eq(band) & plot_data['recording_family'].eq(family),
            'log10_band_power_uv2',
        ].dropna().to_numpy()
        for family in families
    ]
    axis.boxplot(values, tick_labels=families, showfliers=False)
    axis.set_title(f'{band.capitalize()} power')
    axis.set_xlabel('Recording family')
    axis.grid(axis='y', alpha=0.25)
axes[0].set_ylabel('log10 band power (µV²)')
fig.suptitle('Mu and beta spectral power across all recordings and scalp channels')
fig.tight_layout()
plt.show()

## Interpretation and limitations

The saved PSD answers: **how much oscillatory power each electrode contains at each mu/beta frequency over the full recording**. The feature table summarizes total band power, band power relative to 1–40 Hz, the strongest frequency in each band, and mean PSD.

This is feasible for all files, but whole-recording spectra discard timing. In motor-imagery EEG, the most informative effect is often cue-related mu/beta event-related desynchronization or synchronization (ERD/ERS), not simply high average power. For trial-level inference, the better follow-up is to epoch online/acquisition recordings around event markers, reject contaminated trials, compute Morlet-wavelet or short-time Fourier time-frequency power, and compare task windows against a pre-cue baseline. Closed/open-eye baseline files can still provide resting reference spectra, but should not be treated as if they contain task cues.